# 01 — Construir corpus Wikipedia ES

**Proyecto:** Asistente Agentic RAG sobre Mundial 2026
**Curso:** PLN — USFQ

Este notebook descarga artículos de Wikipedia ES sobre el Mundial 2026, selecciones, estadios, historia de Mundiales, jugadores y conceptos clave del fútbol. Cada artículo se guarda como `.md` con frontmatter YAML en `Corpus_Mundial/wikipedia/`.

## 1. Setup

Instalar dependencias (descomenta la línea si es la primera vez).

In [1]:
# !pip install wikipedia-api pyyaml

In [2]:
import re
import time
import unicodedata
from pathlib import Path
from typing import Optional

import wikipediaapi
import yaml

## 2. Configuración

Rutas, cliente Wikipedia y user-agent.

> **Nota sobre el `USER_AGENT`:** NO es un token ni autenticación. Wikimedia exige que cualquier script que use su API se identifique con nombre del proyecto + un contacto. No autentica nada, no usa créditos.

In [3]:
ROOT = Path('..').resolve()
OUTPUT_ROOT = ROOT / 'Corpus_Mundial' / 'wikipedia'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# User-agent para Wikipedia API. NO es token ni autenticación.
# Wikipedia exige que cualquier script se identifique con: 'nombre-proyecto (contacto)'.
# El correo es solo por si Wikipedia necesita contactarte (en la práctica nunca pasa).
# No se guarda en ningún lado, no usa créditos. Cámbialo si quieres.
USER_AGENT = 'Proyecto-PLN-Mundial2026 (iniguez.dev@gmail.com)'

wiki_es = wikipediaapi.Wikipedia(
    user_agent=USER_AGENT,
    language='es',
    extract_format=wikipediaapi.ExtractFormat.WIKI,
)

print(f'Output root: {OUTPUT_ROOT}')
print(f'User-agent:  {USER_AGENT}')

Output root: C:\Users\Administrador\Documents\MaestriaUSFQ\Procesamiento Lenguaje Natural\Proyecto Final\Corpus_Mundial\wikipedia
User-agent:  Proyecto-PLN-Mundial2026 (iniguez.dev@gmail.com)


## 3. Helpers

Funciones para slugificar títulos, escribir frontmatter YAML y guardar artículos.

In [4]:
def slugify(text):
    '''Convierte un titulo a slug kebab-case ASCII.'''
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode()
    text = re.sub(r'[^\w\s-]', '', text).strip().lower()
    return re.sub(r'[\s_]+', '-', text)


def fetch_article(title):
    '''Trae un articulo de Wikipedia ES. Retorna None si no existe.'''
    page = wiki_es.page(title)
    if not page.exists():
        return None
    return {
        'title': page.title,
        'url': page.fullurl,
        'summary': page.summary,
        'text': page.text,
    }


def write_article_md(article, output_dir, tema, tipo='enciclopedico', tags=None):
    '''Escribe un articulo de Wikipedia como .md con frontmatter YAML.'''
    output_dir.mkdir(parents=True, exist_ok=True)
    slug = slugify(article['title'])
    fm = {
        'titulo': article['title'],
        'tema': tema,
        'tipo': tipo,
        'fuente': 'Wikipedia ES',
        'url': article['url'],
        'tags': tags or [],
    }
    yaml_block = yaml.safe_dump(fm, allow_unicode=True, sort_keys=False)
    body = article['text'].strip()
    content = '---\n' + yaml_block + '---\n\n# ' + article['title'] + '\n\n' + body + '\n'
    out_path = output_dir / (slug + '.md')
    out_path.write_text(content, encoding='utf-8')
    return out_path


def process_list(titles, output_subdir, tema, tipo='enciclopedico', tags=None, sleep=0.5, skip_existing=True):
    '''Itera sobre titulos, trae cada uno y lo guarda.

    Idempotente: si ya existe el .md (por slug de input o de articulo final),
    se salta sin re-fetch. Asi Run All es rapido en re-ejecuciones.
    '''
    output_dir = OUTPUT_ROOT / output_subdir
    output_dir.mkdir(parents=True, exist_ok=True)
    ok, missing, skipped = [], [], []
    seen_slugs = set()
    for title in titles:
        # 1) Skip rapido: si el archivo con slug-de-input ya existe, no hacer fetch
        input_slug = slugify(title)
        if skip_existing and (output_dir / (input_slug + '.md')).exists():
            skipped.append(title)
            print('  [SKIP ya existe]', title)
            seen_slugs.add(input_slug)
            continue

        article = fetch_article(title)
        if article is None:
            missing.append(title)
            print('  [MISS no encontrado]', title)
            continue

        # 2) Skip si otro titulo del batch ya escribio este articulo
        # (caso comun: Tarjeta amarilla y Tarjeta roja redirigen al mismo)
        final_slug = slugify(article['title'])
        if final_slug in seen_slugs:
            skipped.append(title)
            print('  [DUP redirect]', title, '->', final_slug + '.md')
            continue

        # 3) Skip si el archivo final ya existe (de un run previo, via redirect)
        if skip_existing and (output_dir / (final_slug + '.md')).exists():
            skipped.append(title)
            seen_slugs.add(final_slug)
            print('  [SKIP via redirect]', title, '->', final_slug + '.md')
            continue

        path = write_article_md(article, output_dir, tema=tema, tipo=tipo, tags=tags)
        seen_slugs.add(final_slug)
        ok.append(title)
        print('  [OK]', title, '->', path.name, '(' + str(len(article['text'])) + ' chars)')
        time.sleep(sleep)
    return {'ok': ok, 'missing': missing, 'skipped': skipped, 'total': len(titles)}

## 4. Listas de artículos por categoría

Puedes ajustar cualquier lista antes de ejecutar. Los títulos deben coincidir exactamente con el de Wikipedia ES (sensible a mayúsculas y tildes).

In [5]:
# === Mundial 2026 — núcleo ===
MUNDIAL_2026 = [
    'Copa Mundial de Fútbol de 2026',
    'Clasificación para la Copa Mundial de Fútbol de 2026',
    'Sorteo de la Copa Mundial de Fútbol de 2026',
    'Plantillas de la Copa Mundial de Fútbol de 2026',
    'Sedes de la Copa Mundial de Fútbol de 2026',
]

In [6]:
# === Países sede ===
PAISES_SEDE = [
    'Estados Unidos en la Copa Mundial de Fútbol de 2026',
    'Canadá en la Copa Mundial de Fútbol de 2026',
    'México en la Copa Mundial de Fútbol de 2026',
]

In [7]:
# === 48 Selecciones clasificadas (o probables) ===
# Formato Wikipedia: 'Selección de fútbol de X'
SELECCIONES = [
    # CONMEBOL
    'Selección de fútbol de Argentina',
    'Selección de fútbol de Brasil',
    'Selección de fútbol de Uruguay',
    'Selección de fútbol de Colombia',
    'Selección de fútbol de Ecuador',
    'Selección de fútbol de Paraguay',
    'Selección de fútbol de Bolivia',
    'Selección de fútbol de Venezuela',
    # UEFA
    'Selección de fútbol de Alemania',
    'Selección de fútbol de España',
    'Selección de fútbol de Francia',
    'Selección de fútbol de Inglaterra',
    'Selección de fútbol de Portugal',
    'Selección de fútbol de Italia',
    'Selección de fútbol de los Países Bajos',
    'Selección de fútbol de Bélgica',
    'Selección de fútbol de Croacia',
    'Selección de fútbol de Suiza',
    'Selección de fútbol de Polonia',
    'Selección de fútbol de Dinamarca',
    'Selección de fútbol de Suecia',
    'Selección de fútbol de Serbia',
    'Selección de fútbol de Turquía',
    'Selección de fútbol de Austria',
    'Selección de fútbol de Noruega',
    'Selección de fútbol de la República Checa',
    # CAF
    'Selección de fútbol de Marruecos',
    'Selección de fútbol de Senegal',
    'Selección de fútbol de Nigeria',
    'Selección de fútbol de Ghana',
    'Selección de fútbol de Camerún',
    'Selección de fútbol de Egipto',
    'Selección de fútbol de Argelia',
    'Selección de fútbol de Túnez',
    'Selección de fútbol de Costa de Marfil',
    # AFC
    'Selección de fútbol de Japón',
    'Selección de fútbol de Corea del Sur',
    'Selección de fútbol de Australia',
    'Selección de fútbol de Irán',
    'Selección de fútbol de Arabia Saudita',
    'Selección de fútbol de Catar',
    # Concacaf (los 3 sede ya están como host pero también selección)
    'Selección de fútbol de los Estados Unidos',
    'Selección de fútbol de Canadá',
    'Selección de fútbol de México',
    'Selección de fútbol de Costa Rica',
    'Selección de fútbol de Panamá',
    'Selección de fútbol de Jamaica',
    'Selección de fútbol de Honduras',
    # OFC
    'Selección de fútbol de Nueva Zelanda',
]

print(f'Total selecciones a procesar: {len(SELECCIONES)}')

Total selecciones a procesar: 49


In [8]:
# === 16 Estadios sede del Mundial 2026 ===
ESTADIOS = [
    # Estados Unidos (11)
    'MetLife Stadium',
    'AT&T Stadium',
    'SoFi Stadium',
    'Lincoln Financial Field',
    'Mercedes-Benz Stadium',
    'Hard Rock Stadium',
    "Levi's Stadium",
    'Lumen Field',
    'Arrowhead Stadium',
    'Gillette Stadium',
    'NRG Stadium',
    # Canadá (2)
    'BMO Field',
    'BC Place',
    # México (3)
    'Estadio Azteca',
    'Estadio Akron',
    'Estadio BBVA',
]

print(f'Total estadios: {len(ESTADIOS)}')

Total estadios: 16


In [9]:
# === Historia de Mundiales (1930–2022, 22 ediciones) ===
HISTORIA = [f'Copa Mundial de Fútbol de {y}' for y in (
    1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974,
    1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014,
    2018, 2022,
)]

print(f'Total ediciones: {len(HISTORIA)}')

Total ediciones: 22


In [10]:
# === Jugadores destacados (actuales + leyendas + nuevas estrellas + tecnicos) ===
JUGADORES = [
    # Actuales (top establecidos)
    'Lionel Messi',
    'Cristiano Ronaldo',
    'Kylian Mbappé',
    'Erling Haaland',
    'Neymar',
    'Luka Modrić',
    'Robert Lewandowski',
    'Jude Bellingham',
    'Pedri',
    'Vinícius Júnior',
    'Mohamed Salah',
    'Sadio Mané',
    'Harry Kane',
    'Christian Pulisic',
    'Alphonso Davies',
    'Hirving Lozano',
    'Federico Valverde',
    'Lautaro Martínez',
    'Rodrygo',
    'Kevin De Bruyne',
    'Karim Benzema',
    'Antoine Griezmann',
    'Bruno Fernandes',
    'Bernardo Silva',
    'Bukayo Saka',
    'Phil Foden',
    'Rafael Leão',
    'Achraf Hakimi',
    'Son Heung-min',
    'Takefusa Kubo',
    # Nuevas estrellas Gen 2026 (jovenes con proyeccion al Mundial)
    'Lamine Yamal',
    'Florian Wirtz',
    'Jamal Musiala',
    'Endrick',
    'Alejandro Garnacho',
    'Eduardo Camavinga',
    'Aurélien Tchouaméni',
    'Cole Palmer',
    'Arda Güler',
    'João Neves',
    'Pau Cubarsí',
    'Warren Zaïre-Emery',
    'Désiré Doué',
    'Estêvão',
    # Tecnicos top (DT de selecciones favoritas al Mundial 2026)
    'Pep Guardiola',
    'Lionel Scaloni',
    'Didier Deschamps',
    'Carlo Ancelotti',
    # Leyendas
    'Pelé',
    'Diego Maradona',
    'Johan Cruyff',
    'Franz Beckenbauer',
    'Michel Platini',
    'Zinedine Zidane',
    'Ronaldo Nazário',
    'Ronaldinho',
    'Bobby Charlton',
    'Eusébio',
    'Lev Yashin',
    'Garrincha',
    'Alfredo Di Stéfano',
    'Ferenc Puskás',
    'Gerd Müller',
    'Paolo Maldini',
    'Roberto Baggio',
    'Carles Puyol',
    'Andrés Iniesta',
    'Xavi Hernández',
]

print(f'Total jugadores/tecnicos: {len(JUGADORES)}')

Total jugadores/tecnicos: 68


In [11]:
# === Conceptos clave del futbol + extras FIFA ===
CONCEPTOS = [
    # Reglas y conceptos del juego
    'Videoarbitraje',
    'Reglas del fútbol',
    'Fuera de juego',  # antes era 'Fuera de juego (fútbol)' que no existe
    'Tarjeta amarilla',
    'Tarjeta roja',
    'Tiempo de descuento',
    'Tanda de penales',
    'Tiempo suplementario',
    'Fase de grupos',
    'Tiro libre (fútbol)',
    'Tiro penal',
    # Organismos
    'Federación Internacional de Fútbol Asociación',
    'International Football Association Board',
    # Extras: contexto FIFA / estadisticas (anexos y competencias paralelas)
    'Copa Mundial de Fútbol Sub-20',
    'Copa FIFA Confederaciones',
    'Confederaciones de la FIFA',
    'Anexo:Goleadores de la Copa Mundial de Fútbol',
    'Anexo:Estadísticas de la Copa Mundial de Fútbol',
]

print(f'Total conceptos + extras: {len(CONCEPTOS)}')

Total conceptos + extras: 18


## 5. Ejecución

Procesar cada categoría. Si Wikipedia no encuentra un título, queda registrado en `missing` para que lo revises manualmente (probable: nombre con tildes distinto, redirección, o artículo aún no existe).

In [12]:
results = {}

print('▶ Procesando Mundial 2026 (núcleo)')
results['mundial-2026'] = process_list(
    MUNDIAL_2026,
    'mundial-2026',
    tema='mundial-2026',
    tags=['mundial-2026', 'core'],
)

▶ Procesando Mundial 2026 (núcleo)
  [SKIP ya existe] Copa Mundial de Fútbol de 2026
  [SKIP ya existe] Clasificación para la Copa Mundial de Fútbol de 2026
  [MISS no encontrado] Sorteo de la Copa Mundial de Fútbol de 2026
  [MISS no encontrado] Plantillas de la Copa Mundial de Fútbol de 2026
  [MISS no encontrado] Sedes de la Copa Mundial de Fútbol de 2026


In [13]:
print('▶ Procesando países sede')
results['paises-sede'] = process_list(
    PAISES_SEDE,
    'paises-sede',
    tema='paises-sede',
    tags=['mundial-2026', 'sede'],
)

▶ Procesando países sede
  [SKIP ya existe] Estados Unidos en la Copa Mundial de Fútbol de 2026
  [SKIP ya existe] Canadá en la Copa Mundial de Fútbol de 2026
  [SKIP ya existe] México en la Copa Mundial de Fútbol de 2026


In [14]:
print('▶ Procesando 48 selecciones')
results['selecciones'] = process_list(
    SELECCIONES,
    'selecciones',
    tema='selecciones',
    tags=['seleccion', 'mundial-2026'],
)

▶ Procesando 48 selecciones
  [SKIP ya existe] Selección de fútbol de Argentina
  [SKIP ya existe] Selección de fútbol de Brasil
  [SKIP ya existe] Selección de fútbol de Uruguay
  [SKIP ya existe] Selección de fútbol de Colombia
  [SKIP ya existe] Selección de fútbol de Ecuador
  [SKIP ya existe] Selección de fútbol de Paraguay
  [SKIP ya existe] Selección de fútbol de Bolivia
  [SKIP ya existe] Selección de fútbol de Venezuela
  [SKIP ya existe] Selección de fútbol de Alemania
  [SKIP ya existe] Selección de fútbol de España
  [SKIP ya existe] Selección de fútbol de Francia
  [SKIP ya existe] Selección de fútbol de Inglaterra
  [SKIP ya existe] Selección de fútbol de Portugal
  [SKIP ya existe] Selección de fútbol de Italia
  [SKIP ya existe] Selección de fútbol de los Países Bajos
  [SKIP ya existe] Selección de fútbol de Bélgica
  [SKIP ya existe] Selección de fútbol de Croacia
  [SKIP ya existe] Selección de fútbol de Suiza
  [SKIP ya existe] Selección de fútbol de Polonia
  [SKIP

In [15]:
print('▶ Procesando 16 estadios sede')
results['estadios'] = process_list(
    ESTADIOS,
    'estadios',
    tema='estadios',
    tags=['estadio', 'mundial-2026'],
)

▶ Procesando 16 estadios sede
  [SKIP ya existe] MetLife Stadium
  [SKIP ya existe] AT&T Stadium
  [SKIP ya existe] SoFi Stadium
  [SKIP ya existe] Lincoln Financial Field
  [SKIP ya existe] Mercedes-Benz Stadium
  [SKIP ya existe] Hard Rock Stadium
  [SKIP ya existe] Levi's Stadium
  [SKIP ya existe] Lumen Field
  [SKIP ya existe] Arrowhead Stadium
  [SKIP ya existe] Gillette Stadium
  [SKIP via redirect] NRG Stadium -> estadio-nrg.md
  [SKIP ya existe] BMO Field
  [SKIP via redirect] BC Place -> estadio-bc-place.md
  [SKIP ya existe] Estadio Azteca
  [SKIP ya existe] Estadio Akron
  [SKIP ya existe] Estadio BBVA


In [16]:
print('▶ Procesando 22 ediciones de Mundial (historia)')
results['historia-mundiales'] = process_list(
    HISTORIA,
    'historia-mundiales',
    tema='historia',
    tipo='enciclopedico',
    tags=['historia', 'mundial'],
)

▶ Procesando 22 ediciones de Mundial (historia)
  [SKIP ya existe] Copa Mundial de Fútbol de 1930
  [SKIP ya existe] Copa Mundial de Fútbol de 1934
  [SKIP ya existe] Copa Mundial de Fútbol de 1938
  [SKIP ya existe] Copa Mundial de Fútbol de 1950
  [SKIP ya existe] Copa Mundial de Fútbol de 1954
  [SKIP ya existe] Copa Mundial de Fútbol de 1958
  [SKIP ya existe] Copa Mundial de Fútbol de 1962
  [SKIP ya existe] Copa Mundial de Fútbol de 1966
  [SKIP ya existe] Copa Mundial de Fútbol de 1970
  [SKIP ya existe] Copa Mundial de Fútbol de 1974
  [SKIP ya existe] Copa Mundial de Fútbol de 1978
  [SKIP ya existe] Copa Mundial de Fútbol de 1982
  [SKIP ya existe] Copa Mundial de Fútbol de 1986
  [SKIP ya existe] Copa Mundial de Fútbol de 1990
  [SKIP ya existe] Copa Mundial de Fútbol de 1994
  [SKIP ya existe] Copa Mundial de Fútbol de 1998
  [SKIP ya existe] Copa Mundial de Fútbol de 2002
  [SKIP ya existe] Copa Mundial de Fútbol de 2006
  [SKIP ya existe] Copa Mundial de Fútbol de 2010
  

In [17]:
print('▶ Procesando jugadores destacados')
results['jugadores'] = process_list(
    JUGADORES,
    'jugadores',
    tema='jugadores',
    tags=['jugador', 'futbol'],
)

▶ Procesando jugadores destacados
  [SKIP ya existe] Lionel Messi
  [SKIP ya existe] Cristiano Ronaldo
  [SKIP ya existe] Kylian Mbappé
  [SKIP ya existe] Erling Haaland
  [SKIP ya existe] Neymar
  [SKIP ya existe] Luka Modrić
  [SKIP ya existe] Robert Lewandowski
  [SKIP ya existe] Jude Bellingham
  [SKIP ya existe] Pedri
  [SKIP ya existe] Vinícius Júnior
  [SKIP ya existe] Mohamed Salah
  [SKIP ya existe] Sadio Mané
  [SKIP ya existe] Harry Kane
  [SKIP ya existe] Christian Pulisic
  [SKIP ya existe] Alphonso Davies
  [SKIP ya existe] Hirving Lozano
  [SKIP ya existe] Federico Valverde
  [SKIP ya existe] Lautaro Martínez
  [SKIP via redirect] Rodrygo -> rodrygo-goes.md
  [SKIP ya existe] Kevin De Bruyne
  [SKIP ya existe] Karim Benzema
  [SKIP ya existe] Antoine Griezmann
  [SKIP ya existe] Bruno Fernandes
  [SKIP ya existe] Bernardo Silva
  [SKIP ya existe] Bukayo Saka
  [SKIP ya existe] Phil Foden
  [SKIP ya existe] Rafael Leão
  [SKIP ya existe] Achraf Hakimi
  [SKIP ya existe] S

In [18]:
print('▶ Procesando conceptos clave')
results['conceptos'] = process_list(
    CONCEPTOS,
    'conceptos',
    tema='conceptos',
    tags=['concepto', 'reglas'],
)

▶ Procesando conceptos clave
  [SKIP via redirect] Videoarbitraje -> arbitro-asistente-de-video.md
  [SKIP ya existe] Reglas del fútbol
  [OK] Fuera de juego -> fuera-de-juego.md (14118 chars)
  [SKIP via redirect] Tarjeta amarilla -> tarjeta-penal.md
  [DUP redirect] Tarjeta roja -> tarjeta-penal.md
  [SKIP ya existe] Tiempo de descuento
  [SKIP ya existe] Tanda de penales
  [SKIP via redirect] Tiempo suplementario -> prorroga-deporte.md
  [MISS no encontrado] Fase de grupos
  [SKIP ya existe] Tiro libre (fútbol)
  [SKIP via redirect] Tiro penal -> penalti.md
  [SKIP via redirect] Federación Internacional de Fútbol Asociación -> fifa.md
  [SKIP ya existe] International Football Association Board
  [OK] Copa Mundial de Fútbol Sub-20 -> copa-mundial-de-futbol-sub-20.md (9795 chars)
  [OK] Copa FIFA Confederaciones -> copa-fifa-confederaciones.md (16260 chars)
  [MISS no encontrado] Confederaciones de la FIFA
  [OK] Anexo:Goleadores de la Copa Mundial de Fútbol -> anexogoleadores-de-la-c

## 6. Resumen

Estadísticas finales y lista de títulos no encontrados (si los hay).

In [19]:
print('=' * 70)
print('RESUMEN')
print('=' * 70)
total_ok = 0
total_skipped = 0
total_missing = 0
for category, stats in results.items():
    n_ok = len(stats['ok'])
    n_skip = len(stats.get('skipped', []))
    n_miss = len(stats['missing'])
    n_tot = stats['total']
    print(f'{category:22s}  OK: {n_ok:3d}  SKIP: {n_skip:3d}  MISS: {n_miss:2d}  TOTAL: {n_tot:3d}')
    total_ok += n_ok
    total_skipped += n_skip
    total_missing += n_miss

print('-' * 70)
print(f'{"TOTAL":22s}  OK: {total_ok:3d}  SKIP: {total_skipped:3d}  MISS: {total_missing:2d}')
print()
print('OK   = descargados en este run')
print('SKIP = ya existian de un run anterior (idempotente)')
print('MISS = no encontrados en Wikipedia ES (titulo erroneo o no existe)')
print()

if total_missing > 0:
    print('Titulos no encontrados:')
    for category, stats in results.items():
        for title in stats['missing']:
            print(f'  [{category}] {title}')

RESUMEN
mundial-2026            OK:   0  SKIP:   2  MISS:  3  TOTAL:   5
paises-sede             OK:   0  SKIP:   3  MISS:  0  TOTAL:   3
selecciones             OK:   0  SKIP:  49  MISS:  0  TOTAL:  49
estadios                OK:   0  SKIP:  16  MISS:  0  TOTAL:  16
historia-mundiales      OK:   0  SKIP:  22  MISS:  0  TOTAL:  22
jugadores               OK:  17  SKIP:  50  MISS:  1  TOTAL:  68
conceptos               OK:   5  SKIP:  11  MISS:  2  TOTAL:  18
----------------------------------------------------------------------
TOTAL                   OK:  22  SKIP: 153  MISS:  6

OK   = descargados en este run
SKIP = ya existian de un run anterior (idempotente)
MISS = no encontrados en Wikipedia ES (titulo erroneo o no existe)

Titulos no encontrados:
  [mundial-2026] Sorteo de la Copa Mundial de Fútbol de 2026
  [mundial-2026] Plantillas de la Copa Mundial de Fútbol de 2026
  [mundial-2026] Sedes de la Copa Mundial de Fútbol de 2026
  [jugadores] Estêvão
  [conceptos] Fase de grupos


In [20]:
# Validar que el output exista
for subdir in OUTPUT_ROOT.iterdir():
    if subdir.is_dir():
        n = len(list(subdir.glob('*.md')))
        print(f'{subdir.name:25s}  {n:3d} archivos .md')

conceptos                   15 archivos .md
estadios                    16 archivos .md
historia-mundiales          22 archivos .md
jugadores                   67 archivos .md
mundial-2026                 2 archivos .md
paises-sede                  3 archivos .md
selecciones                 49 archivos .md
